In [ ]:
import os
import pandas as pd
import numpy as np

input_file = "dataset.csv"

if not os.path.exists(input_file):
    try:
        from google.colab import files
        print(f"File '{input_file}' tidak ditemukan. Silakan upload file:")
        uploaded = files.upload()
        if input_file not in uploaded:
            input_file = list(uploaded.keys())[0]
    except ImportError:
        pass

print(f"Membaca {input_file}...")
df = pd.read_csv(input_file)
print(f"Total baris dataset awal: {len(df)} baris x {df.shape[1]} kolom")

In [ ]:
rename_map = {
    "vp_7000m": "Vp_7000m",
    "vp_6000m": "Vp_6000m",
    "vpvs_60km": "VpVs_60km",
    "dvpvs_60km": "dVpVs_60km"
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

selected_columns = [
    "Vp_7000m", "crustal_thickness", "dVs_20km", "Vp_6000m", "VpVs_60km",
    "dVs_60km", "gg_mantle_60km", "dVpVs_60km", "dVs_40km", "elevation_detrend",
    "T_1000m", "T_2000m", "T_3000m", "T_4000m", "T_5000m",
    "Lat", "Long"
]
print("Penyelarasan nama kolom selesai.")

In [ ]:
if len(df) >= 534841:
    df_101_raw = df.iloc[534841:].copy().reset_index(drop=True)
    df_pool = df.iloc[:534841].copy().reset_index(drop=True)
else:
    df_101_raw = df.tail(101).copy().reset_index(drop=True)
    df_pool = df.iloc[:-101].copy().reset_index(drop=True)

available_cols = [c for c in selected_columns if c in df_101_raw.columns]
df_101_clean = df_101_raw[available_cols]
out_101_clean = "101_dataset_testing.csv"
df_101_clean.to_csv(out_101_clean, index=False)
print(f"[1] Berhasil menyimpan: '{out_101_clean}' (Ukuran: {df_101_clean.shape})")

out_101_full = "101_dataset_testing_full_attributes.csv"
df_101_raw.to_csv(out_101_full, index=False)
print(f"[2] Berhasil menyimpan: '{out_101_full}' (Ukuran: {df_101_raw.shape})")

In [ ]:
available_pool_cols = [c for c in selected_columns if c in df_pool.columns]
df_pool_selected = df_pool[available_pool_cols]

print(f"Mengambil 200.000 sampel acak dari {len(df_pool_selected)} data (random_state=42)...")
if len(df_pool_selected) > 200000:
    df_200k = df_pool_selected.sample(n=200000, random_state=42).reset_index(drop=True)
else:
    df_200k = df_pool_selected.copy()

output_200k = "dataset_200k_with_coords.csv"
df_200k.to_csv(output_200k, index=False)
print(f"[3] Berhasil menyimpan: '{output_200k}' (Ukuran: {df_200k.shape})")

In [ ]:
print("="*60)
print("VERIFIKASI HASIL EKSTRAKSI DAN SAMPLING")
print("="*60)
print(f"- Data Uji Coba Terisolasi (17 Kolom) : {out_101_clean} -> {len(df_101_clean)} baris")
print(f"- Data Uji Coba Lengkap (119 Kolom)   : {out_101_full} -> {len(df_101_raw)} baris")
print(f"- Data Latih dan Evaluasi (17 Kolom)  : {output_200k} -> {len(df_200k)} baris")
print("="*60)
print("\n5 Baris Pertama Data 200k:")
display(df_200k.head())
print("\n5 Baris Pertama Data 101 Uji Coba:")
display(df_101_clean.head())